# Bangkok data preprocess

This notebook prepares hourly AirBKK data as daily ground truth by following `BANGKOK_ENRICHMENT_GUIDE.html`.

Main workflow:

1. Load hourly station data and station metadata
2. Standardize the schema to `station_id`, `lat`, `lon`, `timestamp`, and `temp_c`
3. Convert timestamps to `Asia/Bangkok` and create `date`
4. Retain only temperatures from 15–45°C
5. Spatially join stations to the same boundaries used to create `bangkok-districts-50`
6. Retain station-days with at least 80% of 24 hourly readings, or at least 20 readings
7. Aggregate the data to `name`, `date`, `tmax`, and `tmin`

The notebook does not modify the source file. The final output contains only the four columns specified by the Guide, and missing observations are not filled with zero.

## Step 0 — Imports and file paths

This cell locates the project root from the directory in which Jupyter was opened, allowing the notebook to run from either the project root or the `Bangkok Validation` folder.

Files used:

- Hourly data: `Station Data/AirBKK/AirBKK_weather_hourly_3month_50District.csv`
- Station metadata: `Station Data/AirBKK/AirBKK_station_metadata_plus.xlsx`
- Project boundary: `Thailand Polygon/bangkok_districts.geojson`
- Output: `Bangkok Validation/AirBKK_daily_tmax_tmin_guide.csv`

The Jupyter kernel must include `geopandas`. If it is not installed, run `%pip install geopandas`, restart the kernel, and then run the notebook.

In [1]:
from pathlib import Path
import math
import os

import geopandas as gpd
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Locate the project folder from the current working directory and its parents."""
    for candidate in (start, *start.parents):
        source = candidate / "Station Data" / "AirBKK" / "AirBKK_weather_hourly_3month_50District.csv"
        if source.exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Open the notebook from within the Crisis Ready HeatriskAPI folder."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_CSV = PROJECT_ROOT / "Station Data" / "AirBKK" / "AirBKK_weather_hourly_3month_50District.csv"
METADATA_XLSX = PROJECT_ROOT / "Station Data" / "AirBKK" / "AirBKK_station_metadata_plus.xlsx"
BOUNDARY_GEOJSON = PROJECT_ROOT / "Thailand Polygon" / "bangkok_districts.geojson"

# Use an environment variable only when the output location must be changed during testing
default_output = PROJECT_ROOT / "Bangkok Validation" / "AirBKK_daily_tmax_tmin_guide.csv"
OUTPUT_CSV = Path(os.environ.get("BANGKOK_PREPROCESS_OUTPUT", str(default_output)))

EXPECTED_READINGS_PER_DAY = 24
MIN_COMPLETENESS = 0.8
MIN_READINGS_PER_DAY = math.ceil(EXPECTED_READINGS_PER_DAY * MIN_COMPLETENESS)

print(f"Project root: {PROJECT_ROOT}")
print(f"Minimum readings per station-day: {MIN_READINGS_PER_DAY}")
print(f"Output: {OUTPUT_CSV}")

Project root: D:\Crisis Ready HeatriskAPI
Minimum readings per station-day: 20
Output: D:\Crisis Ready HeatriskAPI\Bangkok Validation\AirBKK_daily_tmax_tmin_guide.csv


## Step 1 — Load hourly data

Check that all required columns are present and that `station_id + timestamp_local` is unique. Stop if duplicates are found so daily completeness is not overstated.

In [2]:
hourly = pd.read_csv(SOURCE_CSV, encoding="utf-8-sig", low_memory=False)

required_hourly_columns = {"station_id", "timestamp_local", "Temp"}
missing_columns = required_hourly_columns.difference(hourly.columns)
if missing_columns:
    raise ValueError(f"Hourly file is missing columns: {sorted(missing_columns)}")

duplicate_count = hourly.duplicated(["station_id", "timestamp_local"]).sum()
if duplicate_count:
    raise ValueError(f"Found {duplicate_count:,} duplicate station_id + timestamp_local rows")

print(f"Hourly rows: {len(hourly):,}")
print(f"Stations: {hourly['station_id'].nunique():,}")
print(f"Duplicate station timestamps: {duplicate_count:,}")
display(hourly.head())

Hourly rows: 112,796
Stations: 50
Duplicate station timestamps: 0


,station_id,date,time,timestamp_local,district_th,district_en,WS,WD,Temp,RH,BP
0,56,2026-06-01,00:00:00,2026-06-01T00:00:00+07:00,เขตดินแดง,Din Daeng district,0.8,245.1,25.5,89.8,1010.8
1,56,2026-06-01,01:00:00,2026-06-01T01:00:00+07:00,เขตดินแดง,Din Daeng district,2.6,6.4,25.7,90.5,1009.8
2,56,2026-06-01,02:00:00,2026-06-01T02:00:00+07:00,เขตดินแดง,Din Daeng district,2.1,0.2,25.8,90.1,1008.5
3,56,2026-06-01,03:00:00,2026-06-01T03:00:00+07:00,เขตดินแดง,Din Daeng district,2.4,4.1,26.0,89.5,1008.0
4,56,2026-06-01,04:00:00,2026-06-01T04:00:00+07:00,เขตดินแดง,Din Daeng district,2.7,33.9,25.9,89.4,1007.7


## Step 2 — Add latitude and longitude from metadata

The Guide requires `lat` and `lon` to assign stations to the same polygons as the project. Because the metadata contains more stations than the 50-district dataset, retain only the `station_id` values present in the hourly file.

This cell stops if coordinates are missing or if the metadata contains duplicate station IDs.

In [3]:
metadata = pd.read_excel(METADATA_XLSX, sheet_name="AirBKK_Plus", engine="openpyxl")

required_metadata_columns = {"Airbkk_station_id", "Latitude", "Longitude"}
missing_metadata_columns = required_metadata_columns.difference(metadata.columns)
if missing_metadata_columns:
    raise ValueError(f"Metadata is missing columns: {sorted(missing_metadata_columns)}")

station_ids = set(hourly["station_id"].unique())
station_metadata = metadata.loc[
    metadata["Airbkk_station_id"].isin(station_ids),
    ["Airbkk_station_id", "Latitude", "Longitude"],
].rename(
    columns={
        "Airbkk_station_id": "station_id",
        "Latitude": "lat",
        "Longitude": "lon",
    }
)

station_metadata["lat"] = pd.to_numeric(station_metadata["lat"], errors="coerce")
station_metadata["lon"] = pd.to_numeric(station_metadata["lon"], errors="coerce")

if station_metadata["station_id"].duplicated().any():
    raise ValueError("Metadata contains duplicate station_id values")
if set(station_metadata["station_id"]) != station_ids:
    missing_ids = sorted(station_ids.difference(station_metadata["station_id"]))
    raise ValueError(f"Metadata has no coordinates for station_id values: {missing_ids}")
if station_metadata[["lat", "lon"]].isna().any().any():
    raise ValueError("Metadata contains missing latitude/longitude values or values that cannot be converted to numbers")

print(f"Matched metadata rows: {len(station_metadata):,}")
display(station_metadata.head())

Matched metadata rows: 50


,station_id,lat,lon
0,56,13.761922,100.558606
1,57,13.691511,100.614570
2,58,13.682136,100.506051
3,59,13.759085,100.534579
4,60,13.776664,100.520982


## Step 3 — Align the schema and timestamps with the Guide

Create a working table containing `station_id`, `lat`, `lon`, `timestamp`, and `temp_c`.

- `timestamp_local` already includes the `+07:00` offset, but it is still parsed through UTC and converted back to `Asia/Bangkok` as specified by the Guide.
- `date` is recreated from the local timestamp rather than using the original date column.
- Blank `Temp` values, `n/a`, `Cal.`, and other text values are converted to `NaN`.

In [5]:
stations = hourly[["station_id", "timestamp_local", "Temp"]].merge(
    station_metadata,
    on="station_id",
    how="left",
    validate="many_to_one",
)

stations = stations.rename(columns={"timestamp_local": "timestamp", "Temp": "temp_c"})
stations["timestamp"] = (
    pd.to_datetime(stations["timestamp"], errors="coerce", utc=True)
    .dt.tz_convert("Asia/Bangkok")
)
stations["date"] = stations["timestamp"].dt.date
stations["temp_c"] = pd.to_numeric(stations["temp_c"], errors="coerce")

invalid_timestamp_count = stations["timestamp"].isna().sum()
missing_temp_count = stations["temp_c"].isna().sum()

print(f"Invalid/missing timestamps: {invalid_timestamp_count:,}")
print(f"Missing/non-numeric temperatures: {missing_temp_count:,}")
display(stations.head(30))

Invalid/missing timestamps: 0
Missing/non-numeric temperatures: 4,193


,station_id,timestamp,temp_c,lat,lon,date
0,56,2026-06-01 00:00:00+07:00,25.5,13.761922,100.558606,2026-06-01
1,56,2026-06-01 01:00:00+07:00,25.7,13.761922,100.558606,2026-06-01
2,56,2026-06-01 02:00:00+07:00,25.8,13.761922,100.558606,2026-06-01
3,56,2026-06-01 03:00:00+07:00,26.0,13.761922,100.558606,2026-06-01
4,56,2026-06-01 04:00:00+07:00,25.9,13.761922,100.558606,2026-06-01
5,56,2026-06-01 05:00:00+07:00,26.0,13.761922,100.558606,2026-06-01
6,56,2026-06-01 06:00:00+07:00,26.0,13.761922,100.558606,2026-06-01
7,56,2026-06-01 07:00:00+07:00,26.2,13.761922,100.558606,2026-06-01
8,56,2026-06-01 08:00:00+07:00,27.2,13.761922,100.558606,2026-06-01
9,56,2026-06-01 09:00:00+07:00,29.0,13.761922,100.558606,2026-06-01


## Step 4 — Physical plausibility: retain only 15–45°C

The Guide requires air temperatures used as ground truth to remain within 15–45°C.

This step removes `NaN`, `Temp=0`, and out-of-range values before counting readings per day.

In [6]:
valid_timestamp = stations["timestamp"].notna()
physically_plausible = stations["temp_c"].between(15, 45)

cleaned = stations.loc[
    valid_timestamp & physically_plausible,
    ["station_id", "lat", "lon", "timestamp", "date", "temp_c"],
].copy()

print(f"Rows before filtering: {len(stations):,}")
print(f"Rows after 15–45°C filter: {len(cleaned):,}")
print(f"Rows removed: {len(stations) - len(cleaned):,}")
print(f"Temperature range retained: {cleaned['temp_c'].min():.1f}–{cleaned['temp_c'].max():.1f}°C")

Rows before filtering: 112,796
Rows after 15–45°C filter: 108,603
Rows removed: 4,193
Temperature range retained: 22.6–39.0°C


## Step 5 — Prepare boundaries that match `bangkok-districts-50`

The source GeoJSON uses `amp_en`, `amp_th`, and `amp_code`. The notebook that created the project converts them to:

```python
name = amp_en
name_th = amp_th
district_code = amp_code
```

Use GeoPandas to read the boundaries and create `name` in the same way as the project, then match station coordinates to districts using `predicate="within"` as specified by the Guide.

In [8]:
boundaries = gpd.read_file(BOUNDARY_GEOJSON)

if len(boundaries) != 50:
    raise ValueError(f"Expected 50 district polygons but found {len(boundaries)}")
required_boundary_columns = {"amp_en", "amp_th", "amp_code", "geometry"}
if not required_boundary_columns.issubset(boundaries.columns):
    raise ValueError("Boundary data is missing columns required to create the project")
if boundaries.crs is None:
    raise ValueError("Boundary data has no CRS")
boundaries = boundaries.to_crs("EPSG:4326")

for source, target in [("amp_en", "name"), ("amp_th", "name_th"), ("amp_code", "district_code")]:
    boundaries[target] = boundaries[source].fillna("").astype(str).str.strip()
if boundaries[["name", "name_th", "district_code"]].eq("").any().any():
    raise ValueError("Every polygon must have amp_en, amp_th, and amp_code")
if boundaries["name"].duplicated().any():
    raise ValueError("Boundary data contains duplicate district names")

print(f"District polygons: {len(boundaries)}")
print("Example project name:", boundaries.iloc[15]["name"])

District polygons: 50
Example project name: Din Daeng


In [9]:
boundaries.head()

,amp_code,amp_th,amp_en,pro_code,pro_th,pro_en,reg_nesdb,reg_royin,perimeter,area_sqkm,geometry,name,name_th,district_code
0,1042,สายไหม,Sai Mai,10,กรุงเทพมหานคร,Bangkok,Central,Central,29.382612,43.193068,"MULTIPOLYGON (((100.68359 13.91154, 100.69222 ...",Sai Mai,สายไหม,1042
1,1043,คันนายาว,Khan Na Yao,10,กรุงเทพมหานคร,Bangkok,Central,Central,24.158310,24.938218,"MULTIPOLYGON (((100.67084 13.85276, 100.68373 ...",Khan Na Yao,คันนายาว,1043
2,1040,บางแค,Bang Khae,10,กรุงเทพมหานคร,Bangkok,Central,Central,29.036152,47.218134,"MULTIPOLYGON (((100.42599 13.71222, 100.42764 ...",Bang Khae,บางแค,1040
3,1041,หลักสี่,Lak Si,10,กรุงเทพมหานคร,Bangkok,Central,Central,21.523015,23.022377,"MULTIPOLYGON (((100.59472 13.8877, 100.58806 1...",Lak Si,หลักสี่,1041
4,1048,ทวีวัฒนา,Thawi Watthana,10,กรุงเทพมหานคร,Bangkok,Central,Central,32.939191,52.034992,"MULTIPOLYGON (((100.40647 13.76014, 100.40631 ...",Thawi Watthana,ทวีวัฒนา,1048


## Step 6 — Spatially join stations to polygon names

Match the metadata coordinates of all 50 stations to polygons, then merge `name` back into the temperature-filtered hourly data. Matching all stations before the completeness filter allows the coverage report to show stations with no usable data as zero accepted days.

This cell stops if any station lies outside all polygons or matches more than one polygon.

In [10]:
points = gpd.GeoDataFrame(
    station_metadata.copy(),
    geometry=gpd.points_from_xy(station_metadata["lon"], station_metadata["lat"]),
    crs="EPSG:4326",
)
station_matches = gpd.sjoin(
    points, boundaries[["name", "geometry"]], how="left", predicate="within"
)
station_polygons = station_matches[["station_id", "name"]].copy()
if station_polygons["name"].isna().any():
    raise ValueError("One or more stations are outside all district polygons")
if station_polygons["station_id"].duplicated().any():
    raise ValueError("The spatial join returned more than one name for a station_id")
if len(station_polygons) != len(station_metadata):
    raise ValueError("The spatial join returned a station count that does not match the metadata")

joined = cleaned.merge(
    station_polygons,
    on="station_id",
    how="left",
    validate="many_to_one",
)
if joined["name"].isna().any():
    raise ValueError("One or more hourly rows have no name after the spatial join")

print(f"Stations matched to polygons: {station_polygons['station_id'].nunique()}")
display(station_polygons.sort_values("name").head())

Stations matched to polygons: 50


,station_id,name
34,90,Bang Bon
20,76,Bang Kapi
48,104,Bang Khae
42,98,Bang Khen
12,68,Bang Kho Laem


## Step 7 — Station-day completeness specified by the Guide

The Guide specifies:

```python
EXPECTED_READINGS_PER_DAY = 24
complete = readings_per_day[readings_per_day >= EXPECTED_READINGS_PER_DAY * 0.8].index
```

`24 × 0.8 = 19.2`; therefore, the minimum whole-number threshold is 20 readings per station-day.

No 80% coverage threshold is applied to a station across the full study period, and missing dates are not filled with zero.

In [14]:
readings_per_day = joined.groupby(["station_id", "date"]).size()
complete = readings_per_day[
    readings_per_day >= EXPECTED_READINGS_PER_DAY * MIN_COMPLETENESS
].index

complete_rows = (
    joined.set_index(["station_id", "date"])
    .loc[complete]
    .reset_index()
)

print(f"Station-days observed after physical filter: {len(readings_per_day):,}")
print(f"Station-days passing >= {MIN_READINGS_PER_DAY}/24: {len(complete):,}")
print(f"Station-days removed by completeness rule: {len(readings_per_day) - len(complete):,}")
print("Readings per accepted station-day:")
display(readings_per_day.loc[complete].value_counts().sort_index().rename("station_days").to_frame())

Station-days observed after physical filter: 4,530
Station-days passing >= 20/24: 4,521
Station-days removed by completeness rule: 9
Readings per accepted station-day:


,station_days
21,1
22,5
23,11
24,4504


In [16]:
complete_rows.count()

station_id    108480
date          108480
lat           108480
lon           108480
timestamp     108480
temp_c        108480
name          108480
dtype: int64

## Step 8 — Aggregate hourly rows into `tmax` and `tmin`

Aggregate by `name + date` exactly as specified by the Guide. The output contains only four columns:

```text
name, date, tmax, tmin
```

A station-day that fails the completeness criterion has no row in the output.

In [19]:
daily = (
    complete_rows.groupby(["name", "date"])["temp_c"]
    .agg(tmax="max", tmin="min")
    .reset_index()
    .sort_values(["name", "date"])
    .reset_index(drop=True)
)

daily = daily[["name", "date", "tmax", "tmin"]]

print(f"Daily rows: {len(daily):,}")
print(f"Districts with at least one accepted day: {daily['name'].nunique():,}")
print(f"Date range: {daily['date'].min()} to {daily['date'].max()}")
display(daily.head(25))

Daily rows: 4,521
Districts with at least one accepted day: 49
Date range: 2026-06-01 to 2026-09-02


,name,date,tmax,tmin
0,Bang Bon,2026-06-01,35.1,25.7
1,Bang Bon,2026-06-02,36.6,27.6
2,Bang Bon,2026-06-03,34.7,27.8
3,Bang Bon,2026-06-04,34.0,29.3
4,Bang Bon,2026-06-05,34.3,29.3
5,Bang Bon,2026-06-06,33.2,29.4
6,Bang Bon,2026-06-07,32.4,26.2
7,Bang Bon,2026-06-08,34.5,28.5
8,Bang Bon,2026-06-09,34.6,27.8
9,Bang Bon,2026-06-10,34.7,26.1


## Step 9 — Final validation

Validate that the output:

- Contains only the columns specified by the Guide
- Contains no duplicate `name + date` pairs
- Contains no null values
- `tmin ≤ tmax`
- Keeps both `tmin` and `tmax` within 15–45°C

In [20]:
expected_output_columns = ["name", "date", "tmax", "tmin"]
if list(daily.columns) != expected_output_columns:
    raise AssertionError(f"Output columns do not match the Guide: {list(daily.columns)}")
if daily.duplicated(["name", "date"]).any():
    raise AssertionError("Output contains duplicate name + date pairs")
if daily.isna().any().any():
    raise AssertionError("Output contains null values")
if not (daily["tmin"] <= daily["tmax"]).all():
    raise AssertionError("Found tmin > tmax")
if not daily["tmin"].between(15, 45).all():
    raise AssertionError("Found tmin outside 15–45°C")
if not daily["tmax"].between(15, 45).all():
    raise AssertionError("Found tmax outside 15–45°C")

print("Final validation passed")
print(f"Rows: {len(daily):,}")
print(f"Columns: {list(daily.columns)}")

Final validation passed
Rows: 4,521
Columns: ['name', 'date', 'tmax', 'tmin']


## Step 10 — Save the CSV

Save with a UTF-8 BOM so Excel displays Thai data and text correctly. The output file contains only `name,date,tmax,tmin`.

In [22]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
daily.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_CSV}")
print(f"Rows written: {len(daily):,}")

Saved: D:\Crisis Ready HeatriskAPI\Bangkok Validation\AirBKK_daily_tmax_tmin_guide.csv
Rows written: 4,521


## Step 11 — Coverage report for reference

This table starts with all 50 stations in the metadata and reports the number of accepted days for each district. A station with no accepted days is shown as zero without adding any columns to the main output file.

In [23]:
coverage_report = (
    complete_rows[["station_id", "name", "date"]]
    .drop_duplicates()
    .groupby(["station_id", "name"])
    .size()
    .rename("accepted_days")
    .reset_index()
    .sort_values(["accepted_days", "name"])
)

all_station_names = station_polygons.merge(
    coverage_report,
    on=["station_id", "name"],
    how="left",
)
all_station_names["accepted_days"] = all_station_names["accepted_days"].fillna(0).astype(int)

print(f"Stations in coverage report: {len(all_station_names)}")
display(all_station_names.sort_values(["accepted_days", "name"]))

Stations in coverage report: 50


,station_id,name,accepted_days
20,76,Bang Kapi,0
15,71,Suan Luang,30
32,88,Phasi Charoen,84
2,58,Rat Burana,89
10,66,Bang Rak,93
28,84,Bangkok Yai,93
18,74,Don Mueang,93
40,96,Lat Phrao,93
30,86,Taling Chan,93
26,82,Thon Buri,93
